# imports & dataset

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
pd.set_option('display.max_columns', None)
import copy
import os
import sklearn
from sklearn.model_selection import train_test_split
from pcntoolkit import NormData
import warnings
import logging
import pcntoolkit.util.output
from PIL import Image
from pcntoolkit import (
    BLR,
    BsplineBasisFunction,
    NormativeModel,
    NormData,
    load_fcon1000,
    plot_centiles_advanced,
    plot_qq,
    plot_ridge,
)


sns.set_style("darkgrid")

# Suppress some annoying warnings and logs
pymc_logger = logging.getLogger("pymc")

pymc_logger.setLevel(logging.WARNING)
pymc_logger.propagate = False

warnings.simplefilter(action="ignore", category=FutureWarning)
pd.options.mode.chained_assignment = None  # default='warn'
pcntoolkit.util.output.Output.set_show_messages(True)


#this code was needed to run the code below without errors
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    # Legacy Python that doesn't verify HTTPS certificates by default
    pass
else:
    # Handle target environment that doesn't support HTTPS verification
    ssl._create_default_https_context = _create_unverified_https_context

print('test')
pd.set_option('display.max_rows', 150)

test


In [2]:
path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data"

euler_nr = pd.read_csv(os.path.join(path, "euler_nr_data.csv"))
datatable = pd.read_csv(os.path.join(path, "Final_data.csv"))

euler_nr = euler_nr[['SubjectID', 'avg_euler_centered_neg_sqrt']].copy()
df_ukb = datatable.merge(euler_nr, how="right")
df_ukb = df_ukb.iloc[:, 1:]
df_ukb = df_ukb.dropna(axis=1)

#these cols had only zeros (left-vessel had 0s in some cases but not all --> may fit a separate model for this)
df_ukb["avg_euler_centered_neg_sqrt"] = df_ukb["avg_euler_centered_neg_sqrt"].astype(float)
df_ukb = df_ukb[df_ukb["avg_euler_centered_neg_sqrt"] < 10.0]

df_ukb = df_ukb.drop(columns=['SubjectID','Year_initial_scan','Birthyear','avg_euler_centered_neg_sqrt','5th-Ventricle','Left-WM-hypointensities','Left-non-WM-hypointensities','Left-vessel','Left-WM-hypointensities','Left-non-WM-hypointensities','Right-WM-hypointensities','Right-non-WM-hypointensities', 'non-WM-hypointensities'])
df_ukb["Dataset"]= "UKB"
first_cols = ["Age", "Sex", "Site", "Dataset"]
other_cols = df_ukb.columns.difference(first_cols, sort=False)
df_ukb = df_ukb[first_cols + list(other_cols)]

df_o3 = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/csv_files/Oasis3_data.csv")

#these cols had only zeros (left-vessel had 0s in some cases but not all --> may fit a separate model for this)
df_o3 = df_o3.drop(columns=['SubjectID','5th-Ventricle','Left-WM-hypointensities','Left-non-WM-hypointensities','Left-vessel','Left-WM-hypointensities','Left-non-WM-hypointensities','Right-WM-hypointensities','Right-non-WM-hypointensities'])

pd.set_option('display.max_columns', None)
df_o3 = df_o3.iloc[:,1:]

#split datasets into alzheimer vs healthy controls
alzheimer_df = df_o3[df_o3["Diagnosis"] >= 1.0]
healthy_controls_o3 = df_o3[df_o3["Diagnosis"] == 0.0]

#remove diagnosis from the Healthy control dataframe
healthy_controls_o3 = healthy_controls_o3.drop(columns=['Diagnosis'])

#order the HC the same way as the UKB dataset
ordering_ukb = list(df_ukb.columns)
healthy_controls_o3["Dataset"] = "OASIS3"
healthy_controls_o3 = healthy_controls_o3[ordering_ukb]
df_o3["Dataset"] = "OASIS3"

df_adni = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/data_prep/ADNI_data.csv")
df_adni = df_adni.iloc[:,1:]
df_adni.replace(to_replace=['F'], value=0.0, inplace=True)
df_adni.replace(to_replace=['M'], value=1.0, inplace=True)
df_adni = df_adni[df_adni["Sex"] != "X"]
df_adni["Age"] = df_adni["Age"].round()
healthy_controls_adni = df_adni[df_adni["Group"] == "CN"]
healthy_controls_adni.drop(columns=["Group"])
healthy_controls_adni["Dataset"] = "ADNI"
healthy_controls_adni = healthy_controls_adni[ordering_ukb]
df_adni["Dataset"] = "ADNI"

/scratch/quirom/slurm_job_53656518/ipykernel_3869296/415121351.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ukb["Dataset"]= "UKB"
/scratch/quirom/slurm_job_53656518/ipykernel_3869296/415121351.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  healthy_controls_o3["Dataset"] = "OASIS3"
/scratch/quirom/slurm_job_53656518/ipykernel_3869296/415121351.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining a

In [ ]:
data_df = pd.concat([df_ukb, healthy_controls_o3, healthy_controls_adni], axis=0)

#removing sites with only 1 participant per sex [train_test_split error]
data_df = data_df[~data_df["Site"].isin([9.0, 10.0, 14.0, 16.0, 19.0, 20.0, 24.0, 27.0, 35.0, 51.0, 57.0, 68.0, 62.0, 67.0, 70.0, 114.0, 130.0, 133.0, 135.0, 136.0, 141.0, 153.0])]

#removing these ages because they had unique site & age combinations which could not be split 
data_df = data_df[data_df["Age"] != 96.0]
data_df = data_df[data_df["Age"] != 91.0]
data_df = data_df[data_df["Age"] != 95.0]
data_df = data_df[data_df["Age"] != 97.0]

demo = data_df.iloc[:,:4]

response_vars = data_df.filter(like='G_front_middle_thickness')   
data_df = pd.concat([demo, response_vars], axis=1)



# model

In [14]:
data = data_df.copy()
covariates = ["Age"]
batch_effects = ["Sex", "Site"]
data_df
data_df = data_df.drop(columns=['Age','Sex','Site','Dataset'])
columns = list(data_df.columns)

response_vars = columns

# create a NormData object
norm_data = NormData.from_dataframe(
    name="BLR_thickness_test2", dataframe=data, covariates=covariates, batch_effects=batch_effects, response_vars=response_vars
)
norm_data.coords

train, test = norm_data.train_test_split([0.8, 0.2], random_state=42)
norm_data.coords

Process: 3869296 - 2026-06-15 12:50:54 - Dataset "Final_BLR_corrected" created.
    - 35378 observations
    - 35378 unique subjects
    - 1 covariates
    - 2 response variables
    - 2 batch effects:
    	Sex (2)
	Site (35)
    


Coordinates:
  * observations       (observations) int64 283kB 0 1 2 3 ... 35375 35376 35377
  * response_vars      (response_vars) <U27 216B 'lh_G_front_middle_thickness...
  * covariates         (covariates) <U3 12B 'Age'
  * batch_effect_dims  (batch_effect_dims) <U4 32B 'Sex' 'Site'

In [15]:
template_blr = BLR(
    name="BLR_thickness_test2",
    basis_function_mean=BsplineBasisFunction(
        degree=3, nknots=5 
    ),  
    fixed_effect=True,  
    heteroskedastic=True,
    warp_name="warpsinharcsinh",  
)

model = NormativeModel(
    template_regression_model=template_blr,
    savemodel=False,
    evaluate_model=True,
    saveresults=True,
    saveplots=True,
    save_dir="/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/BLR/simple_blr",
    inscaler="standardize",
    outscaler="standardize",
)

model.fit_predict(train, test)

Process: 3869296 - 2026-06-15 12:51:21 - Fitting models on 2 response variables.
Process: 3869296 - 2026-06-15 12:51:21 - Fitting model for lh_G_front_middle_thickness.


/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/pcntoolkit/regression_model/blr.py:729: RuntimeWarning: divide by zero encountered in log
  logdetSigma_n = -np.sum(np.log(self.lambda_n_vec))
/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/pcntoolkit/regression_model/blr.py:743: RuntimeWarning: divide by zero encountered in log
  nlZ = nlZ - np.sum(np.log(self.warp.df(y_unwarped, gamma)))


Process: 3869296 - 2026-06-15 12:51:43 - Fitting model for rh_G_front_middle_thickness.


/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.01282e-61): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/pcntoolkit/util/output.py:239: UserWarning: Process: 3869296 - 2026-06-15 12:52:04 - Estimation of posterior distribution failed due to: 
Matrix is not positive definite
  warnings.warn(message)
/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.44658e-61): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.82127e-61): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/home/preclineu/quirom/.conda/envs/norm_vae/l

Process: 3869296 - 2026-06-15 12:52:05 - Making predictions on 2 response variables.
Process: 3869296 - 2026-06-15 12:52:05 - Computing z-scores for 2 response variables.
Process: 3869296 - 2026-06-15 12:52:05 - Computing z-scores for rh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:05 - Computing z-scores for lh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:05 - Computing centiles for 2 response variables.
Process: 3869296 - 2026-06-15 12:52:05 - Computing centiles for rh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:06 - Computing centiles for lh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:06 - Computing log-probabilities for 2 response variables.
Process: 3869296 - 2026-06-15 12:52:06 - Computing log-probabilities for rh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:06 - Computing log-probabilities for lh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:07 - Computing yhat for 2 response v

/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 28302.
  res = hypotest_fun_out(*samples, **kwds)
/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 28302.
  res = hypotest_fun_out(*samples, **kwds)


Process: 3869296 - 2026-06-15 12:52:43 - Dataset "centile" created.
    - 150 observations
    - 150 unique subjects
    - 1 covariates
    - 2 response variables
    - 2 batch effects:
    	Sex (1)
	Site (1)
    
Process: 3869296 - 2026-06-15 12:52:43 - Computing centiles for 2 response variables.
Process: 3869296 - 2026-06-15 12:52:43 - Computing centiles for rh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:43 - Computing centiles for lh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:43 - Harmonizing data on 2 response variables.
Process: 3869296 - 2026-06-15 12:52:43 - Harmonizing data for rh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:43 - Harmonizing data for lh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:46 - Making predictions on 2 response variables.
Process: 3869296 - 2026-06-15 12:52:46 - Computing z-scores for 2 response variables.
Process: 3869296 - 2026-06-15 12:52:46 - Computing z-scores for rh_G_front_mid

/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 7076.
  res = hypotest_fun_out(*samples, **kwds)
/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 7076.
  res = hypotest_fun_out(*samples, **kwds)


Process: 3869296 - 2026-06-15 12:52:55 - Dataset "centile" created.
    - 150 observations
    - 150 unique subjects
    - 1 covariates
    - 2 response variables
    - 2 batch effects:
    	Sex (1)
	Site (1)
    
Process: 3869296 - 2026-06-15 12:52:55 - Computing centiles for 2 response variables.
Process: 3869296 - 2026-06-15 12:52:55 - Computing centiles for rh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:55 - Computing centiles for lh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:55 - Harmonizing data on 2 response variables.
Process: 3869296 - 2026-06-15 12:52:55 - Harmonizing data for rh_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:52:55 - Harmonizing data for lh_G_front_middle_thickness.


<xarray.NormData> Size: 2MB
Dimensions:            (observations: 7076, response_vars: 2, covariates: 1,
                        batch_effect_dims: 2, centile: 5, statistic: 11)
Coordinates:
  * observations       (observations) int64 57kB 1895 23776 18072 ... 28667 5923
  * response_vars      (response_vars) <U27 216B 'lh_G_front_middle_thickness...
  * covariates         (covariates) <U3 12B 'Age'
  * batch_effect_dims  (batch_effect_dims) <U4 32B 'Sex' 'Site'
  * centile            (centile) float64 40B 0.05 0.25 0.5 0.75 0.95
  * statistic          (statistic) <U8 352B 'EXPV' 'MACE' ... 'SMSE' 'ShapiroW'
Data variables:
    subject_ids        (observations) int64 57kB 1895 23776 18072 ... 28667 5923
    Y                  (observations, response_vars) float64 113kB 2.725 ... ...
    X                  (observations, covariates) float64 57kB 65.0 ... 67.0
    batch_effects      (observations, batch_effect_dims) <U7 396kB '0.0' ... ...
    Z                  (observations, response_vars) float64 113kB -0.2876 .....
    centiles           (centile, observations, response_vars) float64 566kB 2...
    logp               (observations, response_vars) float64 113kB -0.7201 .....
    Yhat               (observations, response_vars) float64 113kB 2.75 ... 2...
    statistics         (response_vars, statistic) float64 176B 0.2377 ... 0.9976
    Y_harmonized       (observations, response_vars) float64 113kB 2.725 ... ...
Attributes:
    real_ids:                       False
    is_scaled:                      False
    name:                           Final_BLR_corrected_test
    unique_batch_effects:           {np.str_('Sex'): [np.str_('0.0'), np.str_...
    batch_effect_counts:            defaultdict(<function NormData.register_b...
    covariate_ranges:               {np.str_('Age'): {'mean': np.float64(56.0...
    batch_effect_covariate_ranges:  {np.str_('Sex'): {np.str_('0.0'): {np.str...

In [17]:
import pandas as pd
data = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/BLR/simple_blr/results/statistics_Final_BLR_corrected_train.csv")
data.reset_index(drop=True)

,statistic,lh_G_front_middle_thickness,rh_G_front_middle_thickness
0,EXPV,0.247384,0.233938
1,MACE,0.009236,0.007525
2,MAPE,0.048375,0.047510
3,MSLL,1.502306,1.539924
4,NLL,1.289129,1.295655
5,R2,0.247343,0.233898
6,RMSE,0.169621,0.165891
7,Rho,0.414841,0.398520
8,Rho_p,0.000000,0.000000
9,SMSE,0.752657,0.766102


In [22]:
data_df = pd.concat([df_ukb, healthy_controls_o3, healthy_controls_adni], axis=0)

#removing sites with only 1 participant per sex [train_test_split error]
data_df = data_df[~data_df["Site"].isin([9.0, 10.0, 14.0, 16.0, 19.0, 20.0, 24.0, 27.0, 35.0, 51.0, 57.0, 68.0, 62.0, 67.0, 70.0, 114.0, 130.0, 133.0, 135.0, 136.0, 141.0, 153.0])]

#removing these ages because they had unique site & age combinations which could not be split 
data_df = data_df[data_df["Age"] != 96.0]
data_df = data_df[data_df["Age"] != 91.0]
data_df = data_df[data_df["Age"] != 95.0]
data_df = data_df[data_df["Age"] != 97.0]

demo = data_df.iloc[:,:4]

response_vars = data_df.filter(like='G_front_middle_thickness')   
data_df = pd.concat([demo, response_vars], axis=1)

data_df["mean_G_front_middle_thickness"] = (data_df["lh_G_front_middle_thickness"] + data_df["rh_G_front_middle_thickness"]) / 2
data_df = data_df.drop(columns = ["lh_G_front_middle_thickness", "rh_G_front_middle_thickness"])
data_df

,Age,Sex,Site,Dataset,mean_G_front_middle_thickness
0,60.0,0.0,11025.0,UKB,2.8675
1,60.0,0.0,11025.0,UKB,2.8145
2,62.0,0.0,11025.0,UKB,2.7765
3,47.0,0.0,11025.0,UKB,2.6585
4,45.0,0.0,11025.0,UKB,2.9670
...,...,...,...,...,...
1138,78.0,0.0,941.0,ADNI,2.6230
1140,72.0,1.0,941.0,ADNI,2.3955
1141,71.0,1.0,941.0,ADNI,2.5510
1142,80.0,1.0,941.0,ADNI,2.4370


In [23]:
data = data_df.copy()
covariates = ["Age"]
batch_effects = ["Sex", "Site"]
data_df
data_df = data_df.drop(columns=['Age','Sex','Site','Dataset'])
columns = list(data_df.columns)

response_vars = columns

# create a NormData object
norm_data = NormData.from_dataframe(
    name="BLR_mean_thickness_test2", dataframe=data, covariates=covariates, batch_effects=batch_effects, response_vars=response_vars
)
norm_data.coords

train, test = norm_data.train_test_split([0.8, 0.2], random_state=42)
norm_data.coords

Process: 3869296 - 2026-06-15 12:57:37 - Dataset "BLR_mean_thickness_test2" created.
    - 35378 observations
    - 35378 unique subjects
    - 1 covariates
    - 1 response variables
    - 2 batch effects:
    	Sex (2)
	Site (35)
    


Coordinates:
  * observations       (observations) int64 283kB 0 1 2 3 ... 35375 35376 35377
  * response_vars      (response_vars) <U29 116B 'mean_G_front_middle_thickness'
  * covariates         (covariates) <U3 12B 'Age'
  * batch_effect_dims  (batch_effect_dims) <U4 32B 'Sex' 'Site'

In [24]:
template_blr = BLR(
    name="BLR_thickness_test2",
    basis_function_mean=BsplineBasisFunction(
        degree=3, nknots=5 
    ),  
    fixed_effect=True,  
    heteroskedastic=True,
    warp_name="warpsinharcsinh",  
)

model = NormativeModel(
    template_regression_model=template_blr,
    savemodel=False,
    evaluate_model=True,
    saveresults=True,
    saveplots=True,
    save_dir="/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/BLR/simple_blr",
    inscaler="standardize",
    outscaler="standardize",
)

model.fit_predict(train, test)

Process: 3869296 - 2026-06-15 12:57:58 - Fitting models on 1 response variables.
Process: 3869296 - 2026-06-15 12:57:58 - Fitting model for mean_G_front_middle_thickness.


/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/pcntoolkit/regression_model/blr.py:729: RuntimeWarning: divide by zero encountered in log
  logdetSigma_n = -np.sum(np.log(self.lambda_n_vec))
/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/pcntoolkit/regression_model/blr.py:743: RuntimeWarning: divide by zero encountered in log
  nlZ = nlZ - np.sum(np.log(self.warp.df(y_unwarped, gamma)))


Process: 3869296 - 2026-06-15 12:58:25 - Making predictions on 1 response variables.
Process: 3869296 - 2026-06-15 12:58:25 - Computing z-scores for 1 response variables.
Process: 3869296 - 2026-06-15 12:58:25 - Computing z-scores for mean_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:58:25 - Computing centiles for 1 response variables.
Process: 3869296 - 2026-06-15 12:58:25 - Computing centiles for mean_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:58:26 - Computing log-probabilities for 1 response variables.
Process: 3869296 - 2026-06-15 12:58:26 - Computing log-probabilities for mean_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:58:26 - Computing yhat for 1 response variables.
Process: 3869296 - 2026-06-15 12:58:26 - Computing yhat for mean_G_front_middle_thickness.


/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 28302.
  res = hypotest_fun_out(*samples, **kwds)


Process: 3869296 - 2026-06-15 12:58:44 - Dataset "centile" created.
    - 150 observations
    - 150 unique subjects
    - 1 covariates
    - 1 response variables
    - 2 batch effects:
    	Sex (1)
	Site (1)
    
Process: 3869296 - 2026-06-15 12:58:44 - Computing centiles for 1 response variables.
Process: 3869296 - 2026-06-15 12:58:44 - Computing centiles for mean_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:58:44 - Harmonizing data on 1 response variables.
Process: 3869296 - 2026-06-15 12:58:44 - Harmonizing data for mean_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:58:46 - Making predictions on 1 response variables.
Process: 3869296 - 2026-06-15 12:58:46 - Computing z-scores for 1 response variables.
Process: 3869296 - 2026-06-15 12:58:46 - Computing z-scores for mean_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:58:46 - Computing centiles for 1 response variables.
Process: 3869296 - 2026-06-15 12:58:46 - Computing centiles for mean_G_front_

/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 7076.
  res = hypotest_fun_out(*samples, **kwds)


Process: 3869296 - 2026-06-15 12:58:51 - Dataset "centile" created.
    - 150 observations
    - 150 unique subjects
    - 1 covariates
    - 1 response variables
    - 2 batch effects:
    	Sex (1)
	Site (1)
    
Process: 3869296 - 2026-06-15 12:58:51 - Computing centiles for 1 response variables.
Process: 3869296 - 2026-06-15 12:58:51 - Computing centiles for mean_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 12:58:51 - Harmonizing data on 1 response variables.
Process: 3869296 - 2026-06-15 12:58:51 - Harmonizing data for mean_G_front_middle_thickness.


<xarray.NormData> Size: 1MB
Dimensions:            (observations: 7076, response_vars: 1, covariates: 1,
                        batch_effect_dims: 2, centile: 5, statistic: 11)
Coordinates:
  * observations       (observations) int64 57kB 1895 23776 18072 ... 28667 5923
  * response_vars      (response_vars) <U29 116B 'mean_G_front_middle_thickness'
  * covariates         (covariates) <U3 12B 'Age'
  * batch_effect_dims  (batch_effect_dims) <U4 32B 'Sex' 'Site'
  * centile            (centile) float64 40B 0.05 0.25 0.5 0.75 0.95
  * statistic          (statistic) <U8 352B 'EXPV' 'MACE' ... 'SMSE' 'ShapiroW'
Data variables:
    subject_ids        (observations) int64 57kB 1895 23776 18072 ... 28667 5923
    Y                  (observations, response_vars) float64 57kB 2.799 ... 2...
    X                  (observations, covariates) float64 57kB 65.0 ... 67.0
    batch_effects      (observations, batch_effect_dims) <U7 396kB '0.0' ... ...
    Z                  (observations, response_vars) float64 57kB 0.3122 ... ...
    centiles           (centile, observations, response_vars) float64 283kB 2...
    logp               (observations, response_vars) float64 57kB -0.5963 ......
    Yhat               (observations, response_vars) float64 57kB 2.741 ... 2...
    statistics         (response_vars, statistic) float64 88B 0.256 ... 0.9981
    Y_harmonized       (observations, response_vars) float64 57kB 2.799 ... 2...
Attributes:
    real_ids:                       False
    is_scaled:                      False
    name:                           BLR_mean_thickness_test2_test
    unique_batch_effects:           {np.str_('Sex'): [np.str_('0.0'), np.str_...
    batch_effect_counts:            defaultdict(<function NormData.register_b...
    covariate_ranges:               {np.str_('Age'): {'mean': np.float64(56.0...
    batch_effect_covariate_ranges:  {np.str_('Sex'): {np.str_('0.0'): {np.str...

In [25]:
import pandas as pd
data = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/BLR/simple_blr/results/statistics_BLR_mean_thickness_test2_test.csv")
data.reset_index(drop=True)


,statistic,mean_G_front_middle_thickness
0,EXPV,0.255988
1,MACE,0.009836
2,MAPE,0.044209
3,MSLL,1.562307
4,NLL,1.266601
5,R2,0.255687
6,RMSE,0.155316
7,Rho,0.432084
8,Rho_p,0.000000
9,SMSE,0.744313


In [38]:
predictions = model.predict(test)

Process: 3869296 - 2026-06-15 14:13:43 - Making predictions on 1 response variables.
Process: 3869296 - 2026-06-15 14:13:43 - Computing z-scores for 1 response variables.
Process: 3869296 - 2026-06-15 14:13:43 - Computing z-scores for mean_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 14:13:43 - Computing centiles for 1 response variables.
Process: 3869296 - 2026-06-15 14:13:43 - Computing centiles for mean_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 14:13:43 - Computing log-probabilities for 1 response variables.
Process: 3869296 - 2026-06-15 14:13:43 - Computing log-probabilities for mean_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 14:13:43 - Computing yhat for 1 response variables.
Process: 3869296 - 2026-06-15 14:13:43 - Computing yhat for mean_G_front_middle_thickness.


/home/preclineu/quirom/.conda/envs/norm_vae/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:579: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 7076.
  res = hypotest_fun_out(*samples, **kwds)


Process: 3869296 - 2026-06-15 14:13:48 - Dataset "centile" created.
    - 150 observations
    - 150 unique subjects
    - 1 covariates
    - 1 response variables
    - 2 batch effects:
    	Sex (1)
	Site (1)
    
Process: 3869296 - 2026-06-15 14:13:48 - Computing centiles for 1 response variables.
Process: 3869296 - 2026-06-15 14:13:48 - Computing centiles for mean_G_front_middle_thickness.
Process: 3869296 - 2026-06-15 14:13:48 - Harmonizing data on 1 response variables.
Process: 3869296 - 2026-06-15 14:13:48 - Harmonizing data for mean_G_front_middle_thickness.


In [43]:
from typing import List, Tuple

import numpy as np
import xarray as xr
from scipy import stats  # type: ignore
from scipy.stats import norm
from sklearn.metrics import (explained_variance_score, r2_score,
                             mean_absolute_percentage_error)

from pcntoolkit.dataio.norm_data import NormData
from pcntoolkit.util.output import Output, Warnings


class Evaluator:
    """
    A class for evaluating normative model predictions.

    This class implements various statistics to assess the quality of
    normative model predictions, including correlation coefficients, error metrics,
    and normality tests.

    Attributes
    ----------
    response_vars : List[str]
        List of response variables to evaluate
    """

    def __init__(self) -> None:
        """Initialize the Evaluator."""
        self.response_vars: List[str] = []

    def evaluate(self, data: NormData, statistics: List[str] = []) -> NormData:
        """
        Evaluate model predictions using multiple statistics.

        Parameters
        ----------
        data : NormData
            Data container with predictions and actual values, and yhat

        Returns
        -------
        NormData
            Data container updated with evaluation statistics
        """
        # data["Yhat"] = data.centiles.sel(centile=0.5, method="nearest")
        assert "Yhat" in data.data_vars, "Yhat must be computed before evaluation"
        all_statistics = [
            "Rho", "Rho_p", "R2", "RMSE", "SMSE",
            "MSLL", "MLL", "ShapiroW", "MACE", "MAPE", "EXPV",
            "Skewness", "Kurtosis",
        ]
        if statistics:
            self.statistics = [m for m in all_statistics if m in statistics]

        else:
            self.statistics = all_statistics

        if "ShapiroW" in self.statistics:
            self.evaluate_shapiro_w(data)
        if "R2" in self.statistics:
            self.evaluate_R2(data)
        if "Rho" in self.statistics:
            self.evaluate_rho(data)
        if "RMSE" in self.statistics:
            self.evaluate_rmse(data)
        if "SMSE" in self.statistics:
            self.evaluate_smse(data)
        if "MSLL" in self.statistics:
            self.evaluate_msll(data)
        if "MLL" in self.statistics:
            self.evaluate_mll(data)
        if "MACE" in self.statistics:
            self.evaluate_mace(data)
        if "MAPE" in self.statistics:
            self.evaluate_mape(data)
        if "EXPV" in self.statistics:
            self.evaluate_expv(data)
        if "Skewness" in self.statistics:
            self.evaluate_skew(data)
        if "Kurtosis" in self.statistics:
            self.evaluate_kurt(data)
        return data


    def evaluate_msll(self, data: NormData) -> None:
        """
        Evaluate Mean Standardized Log Loss (MSLL) for model predictions.

        MSLL compares the log loss of the model to that of a simple baseline predictor
        that always predicts the mean of the training data.

        MSLL = MLL_model - MLL_baseline

        Parameters
        ----------
        data : NormData
            Data container with predictions and actual values. Must contain 'y', 'Yhat',
            and standard deviation predictions.
        """
        for responsevar in data.response_var_list:
            resp_predict_data = data.sel(response_vars=responsevar)
            msll = self._evaluate_msll(resp_predict_data)
            data.statistics.loc[{
                "response_vars": responsevar, "statistic": "MSLL"}] = msll

  

    def _evaluate_msll(self, data: NormData) -> float | None:
        """
        Calculate Mean Standardized Log Loss.

        MSLL compares the fitted model's log loss to a baseline Gaussian model.
        Note that both should be computed on the same (scaled) data to ensure 
        proper comparison, as log-likelihoods are scale-dependent.

        Parameters
        ----------
        data : NormData
            Data container with predictions and actual values.

        Returns
        -------
        float | None
            Mean standardized log loss between actual and predicted values
        """
        # Fitted model mean log loss (negative log-likelihood)
        logp = data["logp"].values
        mll_model = -np.mean(logp)

        # Check that the baseline logp is calculated on the scaled data
        if "baseline_logp" not in data:
            print("Cannot compute MSLL because baseline log probability is "
                  "not computed on scaled data.")
            return None

        # Baseline Gaussian model mean log loss (negative log-likelihood)
        baseline_logp = data["baseline_logp"].values
        mll_baseline = -np.mean(baseline_logp)

        # Compute MSLL (mean standardized log loss)
        msll = mll_model - mll_baseline
        return float(msll)


    def empty_statistic(self) -> xr.DataArray:
        return xr.DataArray(
            np.zeros(len(self.response_vars)),
            dims=("response_vars"),
            coords={"response_vars": self.response_vars},
        )

    def prepare(self, responsevar: str) -> None:
        """Prepare the evaluator for a specific response variable."""
        pass

    def reset(self) -> None:
        """Reset the evaluator state."""
        pass

    def n_params(self) -> int:
        """Return the number of parameters in the model."""
        return 0  # Override in subclasses


def compute_baseline_logp(data):
    baseline_logp = xr.zeros_like(data["Y"])
    for responsevar in data.response_vars.values:
        y = (data["Y"].sel(response_vars=responsevar).values.squeeze())
        mu = np.mean(y)
        sigma = np.std(y, ddof=0)
        sigma = max(sigma, 1e-12)
        logp = norm.logpdf(y, loc=mu, scale=sigma)
        baseline_logp.loc[{"response_vars": responsevar}] = logp
    data["baseline_logp"] = baseline_logp
    return data

predictions = compute_baseline_logp(predictions)
evaluator = Evaluator()
evaluated = evaluator.evaluate(data=predictions, statistics=["MSLL"])


In [51]:
stats = evaluated["statistics"].to_dataframe()
stats

statistics
response_vars                 statistic            
mean_G_front_middle_thickness EXPV         0.255988
                              MACE         0.009836
                              MAPE         0.044209
                              MSLL         1.562307
                              NLL          1.266601
                              R2           0.255687
                              RMSE         0.155316
                              Rho          0.432084
                              Rho_p        0.000000
                              SMSE         0.744313
                              ShapiroW     0.998144